# Demand Forecasting — Individual Household Pipeline

This notebook studies the **individual household/device short-term demand-forecasting pipeline** embedded in the Aggregator Tool.

It covers:

1. Fetching 15-minute data for one `device_id`.
2. Feature engineering for individual demand.
3. Recursive BiLSTM model training.
4. Device-specific artifact paths.
5. Forecasting up to 5 hours ahead.
6. Plotting predictions against recent history.
7. Daily evaluation workflow.

In [ ]:
import os
import re
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Optional, Sequence

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

try:
    import asyncpg
except ImportError:
    asyncpg = None

try:
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam
except ImportError:
    Sequential = load_model = LSTM = Dense = Dropout = Bidirectional = EarlyStopping = Adam = None

pd.set_option('display.max_columns', 100)

In [ ]:
@dataclass
class IndividualConfig:
    look_back_short: int = 16       # 4 hours at 15-minute resolution
    min_horizon: int = 4            # 1 hour
    max_horizon: int = 20           # 5 hours
    default_horizon: int = 4
    validation_fraction: float = 0.10
    random_seed: int = 42

CFG = IndividualConfig()
np.random.seed(CFG.random_seed)

DB_SETTINGS = {
    'host': os.getenv('EHS_DB_HOST'),
    'port': int(os.getenv('EHS_DB_PORT', '5432')),
    'user': os.getenv('EHS_DB_USER'),
    'password': os.getenv('EHS_DB_PASSWORD'),
    'database': os.getenv('EHS_DB_NAME'),
}

ARTIFACT_DIR = os.getenv('FORECAST_ARTIFACT_DIR', './forecast_artifacts')
os.makedirs(ARTIFACT_DIR, exist_ok=True)

## 1. Device-specific artifact paths

- `house_models/{device_id}/gru_recursive_short.h5`
- `house_models/{device_id}/scaler_target.pkl`
- `house_models/{device_id}/scaler_features.pkl`
- `house_models/{device_id}/feature_columns.pkl`
- `house_models/{device_id}/mae.pkl`

In [ ]:
def safe_device_id(device_id: str) -> str:
    cleaned = re.sub(r'[^A-Za-z0-9_.-]+', '_', device_id.strip())
    if not cleaned:
        raise ValueError('device_id cannot be empty')
    return cleaned


def device_artifact_paths(device_id: str) -> dict:
    did = safe_device_id(device_id)
    base = os.path.join(ARTIFACT_DIR, 'house_models', did)
    os.makedirs(base, exist_ok=True)
    return {
        'base': base,
        'model': os.path.join(base, 'recursive_short_model.keras'),
        'target_scaler': os.path.join(base, 'scaler_target.pkl'),
        'feature_scaler': os.path.join(base, 'scaler_features.pkl'),
        'feature_columns': os.path.join(base, 'feature_columns.pkl'),
        'mae': os.path.join(base, 'mae.pkl'),
        'metrics': os.path.join(base, 'metrics.pkl'),
    }

## 2. Data access

In [ ]:
def _validate_db_settings(settings: dict) -> None:
    missing = [k for k, v in settings.items() if v in (None, '')]
    if missing:
        raise RuntimeError(f'Missing database environment variables for: {missing}')
    if asyncpg is None:
        raise ImportError('Install asyncpg to fetch data from PostgreSQL: pip install asyncpg')


async def fetch_quarterly_data_for_device(device_id: str) -> pd.DataFrame:
    _validate_db_settings(DB_SETTINGS)
    query = '''
        SELECT quarter, energy_usage / 1000.0 AS energy_usage
        FROM quarterly_data
        WHERE relay = 0 AND device_id = $1
        ORDER BY quarter;
    '''
    conn = await asyncpg.connect(**DB_SETTINGS)
    try:
        rows = await conn.fetch(query, device_id)
    finally:
        await conn.close()
    return pd.DataFrame(rows, columns=['quarter', 'energy_usage'])


def load_device_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {'quarter', 'energy_usage'}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    return df[['quarter', 'energy_usage']].copy()

## 3. Preprocessing and feature engineering

In [ ]:
def prepare_device_time_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['quarter'] = pd.to_datetime(df['quarter'])
    df = df.sort_values('quarter').set_index('quarter')
    df['energy_usage'] = pd.to_numeric(df['energy_usage'], errors='coerce')
    df['energy_usage'] = df['energy_usage'].ffill()
    return df


def add_individual_quarterly_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['rolling_mean_1h'] = df['energy_usage'].rolling(window=4, min_periods=1).mean()
    df['rolling_std_1h'] = df['energy_usage'].rolling(window=4, min_periods=1).std()
    df['lag_15m'] = df['energy_usage'].shift(1)
    df['lag_30m'] = df['energy_usage'].shift(2)
    df['lag_1h'] = df['energy_usage'].shift(4)
    df['lag_3h'] = df['energy_usage'].shift(12)
    df['lag_6h'] = df['energy_usage'].shift(24)

    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    return df.dropna()

In [ ]:
@dataclass
class DevicePreprocessedData:
    df_scaled: pd.DataFrame
    target_scaler: MinMaxScaler
    feature_scaler: MinMaxScaler
    target_col: str
    feature_cols: list[str]
    model_cols: list[str]


def scale_device_features(df_features: pd.DataFrame, fit: bool = True,
                          target_scaler: Optional[MinMaxScaler] = None,
                          feature_scaler: Optional[MinMaxScaler] = None,
                          feature_cols: Optional[list[str]] = None) -> DevicePreprocessedData:
    target_col = 'energy_usage'
    if feature_cols is None:
        feature_cols = [c for c in df_features.columns if c != target_col]
    target_scaler = target_scaler or MinMaxScaler()
    feature_scaler = feature_scaler or MinMaxScaler()

    df_scaled = df_features.copy()
    if fit:
        df_scaled[[target_col]] = target_scaler.fit_transform(df_features[[target_col]])
        df_scaled[feature_cols] = feature_scaler.fit_transform(df_features[feature_cols])
    else:
        df_scaled[[target_col]] = target_scaler.transform(df_features[[target_col]])
        df_scaled[feature_cols] = feature_scaler.transform(df_features[feature_cols])

    model_cols = [target_col] + feature_cols
    return DevicePreprocessedData(df_scaled[model_cols], target_scaler, feature_scaler, target_col, feature_cols, model_cols)


def create_recursive_sequences(data: pd.DataFrame, look_back: int, target_col: str = 'energy_usage'):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data.iloc[i:i + look_back].values)
        y.append(data.iloc[i + look_back][target_col])
    return np.asarray(X), np.asarray(y).reshape(-1, 1)


def chronological_train_val_split(X, y, validation_fraction=0.10):
    split_idx = int(len(X) * (1 - validation_fraction))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

## 4. Individual recursive model

In [ ]:
def build_individual_recursive_lstm(input_shape: tuple[int, int]):
    if Sequential is None:
        raise ImportError('Install tensorflow to build models.')
    model = Sequential([
        Bidirectional(LSTM(200, activation='tanh', return_sequences=True), input_shape=input_shape),
        Bidirectional(LSTM(100, activation='tanh', return_sequences=False)),
        Dropout(0.2),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model


def inverse_target(values: np.ndarray, scaler: MinMaxScaler) -> np.ndarray:
    shape = values.shape
    return scaler.inverse_transform(values.reshape(-1, 1)).reshape(shape)


def regression_metrics(y_true_scaled, y_pred_scaled, target_scaler: MinMaxScaler) -> dict:
    y_true = inverse_target(y_true_scaled, target_scaler).flatten()
    y_pred = inverse_target(y_pred_scaled, target_scaler).flatten()
    mean_usage = float(np.mean(y_true)) if len(y_true) else np.nan
    mae = mean_absolute_error(y_true, y_pred)
    return {
        'mse': float(mean_squared_error(y_true, y_pred)),
        'rmse': float(math.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mae),
        'mape_percent': float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        'r2': float(r2_score(y_true, y_pred)),
        'mean_energy_usage': mean_usage,
        'percentage_mae': float((mae / mean_usage) * 100) if mean_usage else np.nan,
    }

## 5. Training function

In [ ]:
async def train_house_short_term_recursive(device_id: str, df_raw: Optional[pd.DataFrame] = None):
    if df_raw is None:
        df_raw = await fetch_quarterly_data_for_device(device_id)
    if df_raw.empty:
        raise ValueError(f'No data found for device_id={device_id}')

    df = prepare_device_time_index(df_raw)
    df_features = add_individual_quarterly_features(df)

    if len(df_features) <= CFG.look_back_short:
        raise ValueError('Not enough data after feature engineering.')

    prep = scale_device_features(df_features, fit=True)
    X, y = create_recursive_sequences(prep.df_scaled, CFG.look_back_short)
    X_train, X_val, y_train, y_val = chronological_train_val_split(X, y, CFG.validation_fraction)

    model = build_individual_recursive_lstm((X.shape[1], X.shape[2]))
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=8,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        verbose=1,
    )

    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred, prep.target_scaler)

    paths = device_artifact_paths(device_id)
    model.save(paths['model'])
    joblib.dump(prep.target_scaler, paths['target_scaler'])
    joblib.dump(prep.feature_scaler, paths['feature_scaler'])
    joblib.dump(prep.feature_cols, paths['feature_columns'])
    joblib.dump(metrics['mae'], paths['mae'])
    joblib.dump(metrics, paths['metrics'])

    print(f'Saved individual model artifacts under: {paths["base"]}')
    return model, prep, history, metrics

## 6. Loading a trained device model

In [ ]:
def load_house_artifacts(device_id: str):
    paths = device_artifact_paths(device_id)
    if load_model is None:
        raise ImportError('Install tensorflow to load models.')
    model = load_model(paths['model'])
    target_scaler = joblib.load(paths['target_scaler'])
    feature_scaler = joblib.load(paths['feature_scaler'])
    feature_cols = joblib.load(paths['feature_columns'])
    mae = joblib.load(paths['mae']) if os.path.exists(paths['mae']) else None
    prep = DevicePreprocessedData(
        df_scaled=pd.DataFrame(),
        target_scaler=target_scaler,
        feature_scaler=feature_scaler,
        target_col='energy_usage',
        feature_cols=feature_cols,
        model_cols=['energy_usage'] + feature_cols,
    )
    return model, prep, mae

## 7. Recursive forecasting for one household

In [ ]:
def forecast_house_short_term_recursive(model, prep: DevicePreprocessedData, df_raw: pd.DataFrame,
                                        horizon: int = CFG.default_horizon,
                                        mae: Optional[float] = None) -> pd.DataFrame:
    if not (CFG.min_horizon <= horizon <= CFG.max_horizon):
        raise ValueError(f'horizon must be between {CFG.min_horizon} and {CFG.max_horizon}')

    history = prepare_device_time_index(df_raw)[['energy_usage']].copy()
    predictions = []
    timestamps = []

    for step_idx in range(horizon):
        df_features = add_individual_quarterly_features(history)
        df_scaled = scale_device_features(
            df_features,
            fit=False,
            target_scaler=prep.target_scaler,
            feature_scaler=prep.feature_scaler,
            feature_cols=prep.feature_cols,
        ).df_scaled

        if len(df_scaled) < CFG.look_back_short:
            raise ValueError('Not enough data for forecasting after feature engineering.')

        X_last = df_scaled.iloc[-CFG.look_back_short:].values.reshape(1, CFG.look_back_short, -1)
        pred_scaled = model.predict(X_last, verbose=0)[0, 0]
        pred = prep.target_scaler.inverse_transform([[pred_scaled]])[0, 0]

        next_ts = history.index[-1] + pd.Timedelta(minutes=15)
        history.loc[next_ts, 'energy_usage'] = pred
        timestamps.append(next_ts)
        predictions.append(pred)

    out = pd.DataFrame({'timestamp': timestamps, 'forecasted_energy_usage': predictions})
    if mae is not None:
        out['lower_bound'] = out['forecasted_energy_usage'] - mae
        out['upper_bound'] = out['forecasted_energy_usage'] + mae
    return out

## 8. Plotting

In [ ]:
def plot_house_forecast(df_raw: pd.DataFrame, forecast_df: pd.DataFrame, recent_points: int = 96,
                        title: str = 'Individual household short-term forecast'):
    hist = prepare_device_time_index(df_raw).tail(recent_points)
    plt.figure(figsize=(12, 5))
    plt.plot(hist.index, hist['energy_usage'], label='Recent actual')
    plt.plot(pd.to_datetime(forecast_df['timestamp']), forecast_df['forecasted_energy_usage'], marker='o', label='Forecast')
    if {'lower_bound', 'upper_bound'} <= set(forecast_df.columns):
        plt.fill_between(pd.to_datetime(forecast_df['timestamp']), forecast_df['lower_bound'], forecast_df['upper_bound'], alpha=0.2, label='MAE band')
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Energy usage (kWh)')
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_training_history(history):
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Training loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation loss')
    plt.title('Training history')
    plt.xlabel('Epoch')
    plt.ylabel('MSE loss')
    plt.legend()
    plt.grid(True)
    plt.show()

## 9. Daily evaluation pattern

In [ ]:
def rolling_origin_daily_eval(model, prep: DevicePreprocessedData, df_raw: pd.DataFrame,
                              horizon: int = CFG.default_horizon,
                              origins_per_day: int = 4) -> pd.DataFrame:
    df = prepare_device_time_index(df_raw)
    results = []
    min_history = CFG.look_back_short + 24 

    candidate_origins = df.index[-(96 + horizon):-horizon]
    if len(candidate_origins) == 0:
        raise ValueError('Not enough data for daily evaluation.')

    if origins_per_day and len(candidate_origins) > origins_per_day:
        idx = np.linspace(0, len(candidate_origins) - 1, origins_per_day).astype(int)
        candidate_origins = candidate_origins[idx]

    for origin in candidate_origins:
        hist = df.loc[:origin].copy().reset_index().rename(columns={'index': 'quarter'})
        if len(hist) < min_history:
            continue
        fcst = forecast_house_short_term_recursive(model, prep, hist, horizon=horizon)
        actual = df.loc[pd.to_datetime(fcst['timestamp']), 'energy_usage'] if set(pd.to_datetime(fcst['timestamp'])).issubset(set(df.index)) else None
        if actual is None:
            continue
        for ts, pred, act in zip(fcst['timestamp'], fcst['forecasted_energy_usage'], actual.values):
            results.append({'origin': origin, 'timestamp': ts, 'prediction': pred, 'actual': act, 'error': pred - act})

    return pd.DataFrame(results)